##### Imports

In [ ]:
import sys
import os
import subprocess
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import mcp.client.stdio as _mcp_stdio

In [ ]:
llm = ChatAnthropic(model='claude-sonnet-4-5-20250929', anthropic_api_key="")

##### Testing Repo's

In [ ]:
# Step 1: Open a real file (has a real fileno() — no crash)
_MCP_ERRLOG = open("mcp_server_errors.log", "w", buffering=1)
# Step 2: Save the original stdio_client function
_original_stdio_client = _mcp_stdio.stdio_client
# Step 3: Create a wrapper that forces errlog = our real file
def _patched_stdio_client(server, errlog=None):
    # Always use our real file, ignoring whatever Jupyter passes
    return _original_stdio_client(server, errlog=_MCP_ERRLOG)
# Step 4: Replace MCP's function with our patched version
_mcp_stdio.stdio_client = _patched_stdio_client
# Step 5: Also patch the reference inside the sessions module
#         (langchain-mcp-adapters imports stdio_client separately)
import langchain_mcp_adapters.sessions as _sessions
_sessions.stdio_client = _patched_stdio_client
print("MCP stdio_client patched successfully!")
print(f" MCP subprocess errors will be logged to: {os.path.abspath('mcp_server_errors.log')}")
print(f" Working directory : {os.getcwd()}")
print(f" Python executable : {sys.executable}")
if os.path.exists("math_server.py"):
    print("math_server.py found!")
else:
    print("math_server.py NOT found — save it in:", os.getcwd())
if os.path.exists("weather_server.py"):
    print("weather_server.py found!")
else:
    print("weather_server.py NOT found — save it in:", os.getcwd())

#### Setting Server

In [ ]:
mcp = MultiServerMCPClient({
    "math": {
        "command": sys.executable,     # Full Python path e.g. C:\anaconda3\python.exe
        "args": ["Tools/math_server.py"],    # Must be in same folder as notebook
        "transport": "stdio",          # Communicate via stdin/stdout pipes
    }
})

tools_local = await mcp.get_tools(server_name="math")
tools = await mcp.get_tools()
print("Connected to LOCAL Math MCP Server!")
print(f"All available Tools : {[t.name for t in tools_local]}")

In [ ]:
async def run_agent(question, tools, label=""):
    """Send a question to Claude, let it pick and call an MCP tool."""

    llm_bound = llm.bind_tools(tools)
    messages = [HumanMessage(question)]

    print(f"  {label}")
    print(f"Question: {question}")

    # Step 1: Claude reads question and decides which tool to call
    ai_msg = llm_bound.invoke(messages)
    messages.append(ai_msg)

    # If Claude answered directly without a tool
    if not ai_msg.tool_calls:
        print(f"Claude answered directly: {ai_msg.content}")
        return ai_msg.content

    # Step 2: Execute every tool Claude requested
    for tc in ai_msg.tool_calls:
        print(f"Tool chosen  : {tc['name']}")
        print(f"Arguments    : {tc['args']}")

        # Find the matching tool and call it on the MCP server
        tool_obj = next(t for t in tools if t.name == tc["name"])
        result = await tool_obj.ainvoke(tc["args"])   # ← await works here: inside async def

        print(f"Tool result  : {result}")
        messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    # Step 3: Claude writes the final human-readable answer
    final = llm_bound.invoke(messages)
    print(f"Final Answer : {final.content}")
    return final.content


# ---- Test the math agent ----
# await works here too: Jupyter top-level supports it directly
await run_agent("What is 48 multiplied by 7?",  tools_local, "LOCAL MCP — Math")
await run_agent("What is 144 divided by 12?",   tools_local, "LOCAL MCP — Math")
await run_agent("Subtract 39 from 100",         tools_local, "LOCAL MCP — Math")

In [ ]:
print(os.getcwd())

In [ ]:
mcp_remote = MultiServerMCPClient({
    "weather": {
        "url": "http://127.0.0.1:8000/mcp",   # ← use 127.0.0.1
        "transport": "streamable_http",
    }
})
tools_remote = await mcp_remote.get_tools()
print("Connected to REMOTE Weather MCP Server!")
print(f"Tools available: {[t.name for t in tools_remote]}")
# Expected: ['get_current_weather', 'get_weather_forecast']

In [ ]:
await run_agent("What is the weather in Tokyo?",              tools_remote, "REMOTE MCP — Weather")
await run_agent("Give me a 3-day forecast for London",        tools_remote, "REMOTE MCP — Weather")
await run_agent("Is it sunny in Paris right now?",            tools_remote, "REMOTE MCP — Weather")

In [ ]:
mcp_both = MultiServerMCPClient({
    # LOCAL server — auto-launched as subprocess
    "math": {
        "command": sys.executable,
        "args": ["Tools/math_server.py"],
        "transport": "stdio",
    },
    # REMOTE server — already running, connect via HTTP
    "weather": {
        "url": "http://localhost:8000/mcp",
        "transport": "streamable_http",
    }
})

all_tools = await mcp_both.get_tools()

print("Connected to BOTH servers!")
print(f"All tools: {[t.name for t in all_tools]}")
# Expected: ['add','subtract','multiply','divide',
#            'get_current_weather','get_weather_forecast']

In [ ]:
# Math → goes to LOCAL server
await run_agent("What is 25 multiplied by 8?",          all_tools, "COMBINED — Math")

# Weather → goes to REMOTE server
await run_agent("What is the weather in Dubai?",        all_tools, "COMBINED — Weather")

# Math again
await run_agent("Add 356 and 789",                      all_tools, "COMBINED — Math")

# Weather forecast
await run_agent("3-day weather forecast for New York",  all_tools, "COMBINED — Weather")

In [ ]:
print("Contents of mcp_server_errors.log:")
with open("mcp_server_errors.log", "r") as f:
    content = f.read().strip()
    print(content if content else "(empty — no errors!)")

In [ ]:
import asyncio

async def task_one():
    print("Task 1: started")
    await asyncio.sleep(3)      # pretend: waiting for MCP server
    print("Task 1: finished")

async def task_two():
    print("Task 2: started")
    await asyncio.sleep(1)      # pretend: waiting for Claude
    print("Task 2: finished")

async def main():
    # Run both tasks AT THE SAME TIME
    await asyncio.gather(task_one(), task_two())

await main()